In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import joblib
import os

df = pd.read_csv(r"C:\Users\sksou\fraud-detection\data\creditcard.csv")

print("Dataset shape:", df.shape)
print("Fraud cases:", df['Class'].sum(), f"({df['Class'].mean()*100:.2f}%)")

# ── Feature Engineering (must match backend create_features exactly) ──────────
USER_AVG = 500
USER_HOUR = 14

df['hour'] = df['Time'] % 24
df['is_high_amount']       = (df['Amount'] > 1000).astype(int)
df['is_night']             = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)
df['transaction_velocity'] = df['Amount'] / (df['Time'] + 1)
df['avg_amount']           = df['Amount'].rolling(20, min_periods=1).mean()
df['amount_deviation']     = df['Amount'] - df['avg_amount']
df['amount_dev_user']      = df['Amount'] - USER_AVG
df['time_dev_user']        = abs(df['hour'] - USER_HOUR)
df['is_unusual_time']      = (df['time_dev_user'] > 6).astype(int)
df['anomaly_score']        = abs(df['amount_dev_user']) / (USER_AVG + 1)

FEATURES = [
    'Amount',
    'hour',
    'is_high_amount',
    'is_night',
    'transaction_velocity',
    'avg_amount',
    'amount_deviation',
    'amount_dev_user',
    'time_dev_user',
    'is_unusual_time',
    'anomaly_score',
]

X = df[FEATURES]
y = df['Class']

print("\nFeature matrix shape:", X.shape)
print("Features:", FEATURES)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Model Training ─────────────────────────────────────────────────────────────
print("\nTraining XGBoost...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    verbosity=0,
)
xgb.fit(X_train_scaled, y_train)

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train_scaled, y_train)

print("Training Neural Network...")
nn = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=50,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
nn.fit(X_train_scaled, y_train)

# ── Evaluation ─────────────────────────────────────────────────────────────────
print("\n=== Evaluation ===")

for name, model in [("XGBoost", xgb), ("RandomForest", rf), ("NeuralNet", nn)]:
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    auc   = roc_auc_score(y_test, probs)
    print(f"\n{name}  (AUC: {auc:.4f})")
    print(classification_report(y_test, preds, target_names=["Normal", "Fraud"]))

# ── Save ────────────────────────────────────────────────────────────────────────
save_dir = r"C:\Users\sksou\fraud-detection\models"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(xgb,    os.path.join(save_dir, "xgb.pkl"))
joblib.dump(rf,     os.path.join(save_dir, "rf.pkl"))
joblib.dump(nn,     os.path.join(save_dir, "nn.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))

print("\n✅ All models saved to", save_dir)
print("Feature count:", len(FEATURES), "— backend must match exactly")

Dataset shape: (284807, 31)
Fraud cases: 492 (0.17%)

Feature matrix shape: (284807, 11)
Features: ['Amount', 'hour', 'is_high_amount', 'is_night', 'transaction_velocity', 'avg_amount', 'amount_deviation', 'amount_dev_user', 'time_dev_user', 'is_unusual_time', 'anomaly_score']

Training XGBoost...
Training Random Forest...
Training Neural Network...

=== Evaluation ===

XGBoost  (AUC: 0.7020)
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97     56864
       Fraud       0.01      0.34      0.02        98

    accuracy                           0.95     56962
   macro avg       0.50      0.64      0.50     56962
weighted avg       1.00      0.95      0.97     56962


RandomForest  (AUC: 0.6773)
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.43      0.03      0.06        98

    accuracy                           1.00     56962
   macro avg       0.71      0

C:\Users\sksou\fraud-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sksou\fraud-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sksou\fraud-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape


✅ All models saved to C:\Users\sksou\fraud-detection\models
Feature count: 11 — backend must match exactly
